In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report, precision_score, recall_score

# Load data and predictions
dev_df = pd.read_csv('dev.csv')
dev_preds = []
with open('dev.txt', 'r') as f:
    for line in f:
        dev_preds.append(int(line.strip()))

dev_df['pred'] = dev_preds
true_labels = dev_df['label'].values
pred_labels = np.array(dev_preds)

# OVERALL METRICS

print("=" * 50)
print("OVERALL METRICS")
print("=" * 50)
print(f"F1:        {f1_score(true_labels, pred_labels):.4f}")
print(f"Precision: {precision_score(true_labels, pred_labels):.4f}")
print(f"Recall:    {recall_score(true_labels, pred_labels):.4f}")
print(f"Accuracy:  {accuracy_score(true_labels, pred_labels):.4f}")

# CONFUSION MATRIX

print("\n" + "=" * 50)
print("CONFUSION MATRIX")
print("=" * 50)
cm = confusion_matrix(true_labels, pred_labels)
print(f"TN: {cm[0][0]}  FP: {cm[0][1]}")
print(f"FN: {cm[1][0]}  TP: {cm[1][1]}")
print("\n" + classification_report(true_labels, pred_labels, target_names=['Not Patronizing', 'Patronizing']))

# ERROR ANALYSIS BY KEYWORD
print("=" * 50)
print("PERFORMANCE BY KEYWORD")
print("=" * 50)
for kw in sorted(dev_df['keyword'].unique()):
    subset = dev_df[dev_df['keyword'] == kw]
    pos_count = subset['label'].sum()
    total = len(subset)
    if pos_count > 0:
        kw_f1 = f1_score(subset['label'], subset['pred'])
        kw_prec = precision_score(subset['label'], subset['pred'], zero_division=0)
        kw_rec = recall_score(subset['label'], subset['pred'], zero_division=0)
        print(f"{kw:30s} | F1: {kw_f1:.3f} | Prec: {kw_prec:.3f} | Rec: {kw_rec:.3f} | Pos: {pos_count}/{total}")
    else:
        fp = ((subset['pred'] == 1)).sum()
        print(f"{kw:30s} | No positive labels | FP: {fp}/{total}")

#ERROR ANALYSIS BY SEVERITY
if 'orig_label' in dev_df.columns or 'original_label' in dev_df.columns:
    sev_col = 'orig_label' if 'orig_label' in dev_df.columns else 'original_label'
    dev_df[sev_col] = dev_df[sev_col].astype(float).astype(int)

    print("\n" + "=" * 50)
    print("ERROR ANALYSIS BY SEVERITY")
    print("=" * 50)
    for sev in sorted(dev_df[sev_col].unique()):
        subset = dev_df[dev_df[sev_col] == sev]
        correct = (subset['label'] == subset['pred']).sum()
        total = len(subset)

        if sev >= 2:  # positive class
            caught = subset['pred'].sum()
            print(f"Severity {sev} | Correctly caught: {caught}/{total} ({caught/total*100:.1f}%) | These are TRUE positives in binary")
        else:  # negative class
            fp = subset['pred'].sum()
            print(f"Severity {sev} | False positives: {fp}/{total} ({fp/total*100:.1f}%) | These are TRUE negatives in binary")

# FN EXAMPLES
print("\n" + "=" * 50)
print("FALSE NEGATIVES (missed patronizing)")
print("=" * 50)
fn = dev_df[(dev_df['label'] == 1) & (dev_df['pred'] == 0)]
print(f"Total false negatives: {len(fn)}")
if 'orig_label' in dev_df.columns or 'original_label' in dev_df.columns:
    sev_col = 'orig_label' if 'orig_label' in dev_df.columns else 'original_label'
    print(f"Severity distribution of FNs: {fn[sev_col].value_counts().to_dict()}")
print("\nExamples:")
for i, row in fn.head(5).iterrows():
    print(f"\n  Keyword: {row['keyword']}")
    print(f"  Text: {row['paragraph'][:200]}...")
    if sev_col in dev_df.columns:
        print(f"  Severity: {row[sev_col]}")

# FP EXAMPLES
print("\n" + "=" * 50)
print("FALSE POSITIVES (incorrectly flagged)")
print("=" * 50)
fp_df = dev_df[(dev_df['label'] == 0) & (dev_df['pred'] == 1)]
print(f"Total false positives: {len(fp_df)}")
if 'orig_label' in dev_df.columns or 'original_label' in dev_df.columns:
    print(f"Severity distribution of FPs: {fp_df[sev_col].value_counts().to_dict()}")
print("\nExamples:")
for i, row in fp_df.head(5).iterrows():
    print(f"\n  Keyword: {row['keyword']}")
    print(f"  Text: {row['paragraph'][:200]}...")
    if sev_col in dev_df.columns:
        print(f"  Severity: {row[sev_col]}")

OVERALL METRICS
F1:        0.6364
Precision: 0.6073
Recall:    0.6683
Accuracy:  0.9274

CONFUSION MATRIX
TN: 1809  FP: 86
FN: 66  TP: 133

                 precision    recall  f1-score   support

Not Patronizing       0.96      0.95      0.96      1895
    Patronizing       0.61      0.67      0.64       199

       accuracy                           0.93      2094
      macro avg       0.79      0.81      0.80      2094
   weighted avg       0.93      0.93      0.93      2094

PERFORMANCE BY KEYWORD
disabled                       | F1: 0.667 | Prec: 0.800 | Rec: 0.571 | Pos: 14/194
homeless                       | F1: 0.688 | Prec: 0.629 | Rec: 0.759 | Pos: 29/212
hopeless                       | F1: 0.635 | Prec: 0.541 | Rec: 0.769 | Pos: 26/217
immigrant                      | F1: 0.444 | Prec: 1.000 | Rec: 0.286 | Pos: 7/218
in-need                        | F1: 0.769 | Prec: 0.667 | Rec: 0.909 | Pos: 33/226
migrant                        | F1: 0.500 | Prec: 0.667 | Rec: 0.400 | P